In [ ]:
# Classificação de Gestos EMG - GRABMyo
# Modelo: CNN + LSTM
# Sessões: S1, S2 e S3

import os
import re
import gc
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt, iirnotch
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

BASE_DIR   = '/home/jovyan/work/Untitled Folder/Teste_1 - CNN/Dados'
SESSOES    = ['S1', 'S2', 'S3']
CANAIS_IDX = list(range(16))
N_CANAIS   = 16
WIN_SIZE   = 512
WIN_STEP   = 512
N_CLASSES  = 16
N_FOLDS    = 3
EPOCHS     = 10
BATCH_SIZE = 32
LR         = 1e-3


def extrair_label(nome):
    m = re.search(r'gesture(\d+)', nome, re.IGNORECASE)
    return int(m.group(1)) - 1 if m else None


def filtrar_canal(sinal, fs):
    if len(sinal) < 4 * fs:
        return sinal
    nyq  = fs / 2.0
    low  = 20.0 / nyq
    high = min(450.0, nyq * 0.95) / nyq
    if 0.01 < low < high < 0.99:
        b, a = butter(4, [low, high], btype='band')
        sinal = filtfilt(b, a, sinal)
    w0 = 60.0 / nyq
    if 0.01 < w0 < 0.99:
        b_n, a_n = iirnotch(w0, Q=30)
        sinal = filtfilt(b_n, a_n, sinal)
    return sinal


def carregar_sessao(data_dir):
    arquivos = sorted([f for f in os.listdir(data_dir) if f.endswith('.hea')])
    X_list, y_list = [], []
    for i, fname in enumerate(arquivos):
        nome  = os.path.splitext(fname)[0]
        label = extrair_label(nome)
        if label is None or not (0 <= label < N_CLASSES):
            continue
        try:
            rec = wfdb.rdrecord(os.path.join(data_dir, nome))
            if rec.p_signal is None:
                continue
            sinal    = rec.p_signal[:, CANAIS_IDX].astype(np.float32)
            filtrado = np.zeros_like(sinal)
            for c in range(sinal.shape[1]):
                filtrado[:, c] = filtrar_canal(sinal[:, c], int(rec.fs))
            del sinal
            inicio = 0
            janelas = []
            while inicio + WIN_SIZE <= filtrado.shape[0]:
                janelas.append(filtrado[inicio:inicio + WIN_SIZE, :])
                inicio += WIN_STEP
            del filtrado
            if janelas:
                X_list.append(np.array(janelas, dtype=np.float16))
                y_list.append(np.full(len(janelas), label, dtype=np.int32))
        except Exception:
            pass
        if (i + 1) % 500 == 0:
            gc.collect()
    X = np.concatenate(X_list)
    y = np.concatenate(y_list)
    del X_list, y_list
    gc.collect()
    return X, y


def build_model(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 7, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(128, 5, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(optimizer=keras.optimizers.Adam(LR),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def calcular_auc(y_true, y_prob):
    classes = np.unique(y_true)
    if len(classes) < 2:
        return float('nan')
    y_bin = label_binarize(y_true, classes=np.arange(N_CLASSES))
    try:
        return roc_auc_score(y_bin[:, classes], y_prob[:, classes],
                             multi_class='ovr', average='macro')
    except Exception:
        return float('nan')


# ── carrega apenas os labels de todas as sessoes para montar o kfold ──
# labels sao pequenos (int32), cabem facil na RAM
print('Carregando labels...')
y_parts = []
tamanhos = []
for s in SESSOES:
    data_dir = os.path.join(BASE_DIR, s)
    arquivos = sorted([f for f in os.listdir(data_dir) if f.endswith('.hea')])
    y_list = []
    for fname in arquivos:
        nome  = os.path.splitext(fname)[0]
        label = extrair_label(nome)
        if label is None or not (0 <= label < N_CLASSES):
            continue
        try:
            rec = wfdb.rdrecord(os.path.join(data_dir, nome))
            if rec.p_signal is None:
                continue
            n_janelas = max(0, (rec.p_signal.shape[0] - WIN_SIZE) // WIN_STEP + 1)
            if n_janelas > 0:
                y_list.append(np.full(n_janelas, label, dtype=np.int32))
        except Exception:
            pass
    y_s = np.concatenate(y_list)
    y_parts.append(y_s)
    tamanhos.append(len(y_s))
    print(f'  {s}: {len(y_s)} janelas')

y_all    = np.concatenate(y_parts)
cumsum   = np.cumsum([0] + tamanhos)
n_total  = len(y_all)
print(f'Total: {n_total} janelas')


def carregar_indices(indices):
    """Carrega so as janelas dos indices pedidos, sessao por sessao."""
    X_list, y_list = [], []
    for s_idx, s in enumerate(SESSOES):
        # indices globais que pertencem a essa sessao
        mask      = (indices >= cumsum[s_idx]) & (indices < cumsum[s_idx + 1])
        idx_local = indices[mask] - cumsum[s_idx]
        if len(idx_local) == 0:
            continue
        X_s, y_s = carregar_sessao(os.path.join(BASE_DIR, s))
        X_list.append(X_s[idx_local])
        y_list.append(y_s[idx_local])
        del X_s, y_s
        gc.collect()
    X = np.concatenate(X_list).astype(np.float32)
    y = np.concatenate(y_list)
    del X_list, y_list
    gc.collect()
    return X, y


# treinamento
skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
results    = []
histories  = []
all_y_true = []
all_y_pred = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.arange(n_total), y_all)):
    print(f'\n---- Fold {fold+1}/{N_FOLDS} ----')
    print(f'Carregando treino ({len(tr_idx)} amostras)...')
    X_tr, y_tr = carregar_indices(tr_idx)

    print(f'Carregando validacao ({len(va_idx)} amostras)...')
    X_va, y_va = carregar_indices(va_idx)

    sc = StandardScaler()
    _, w, ch = X_tr.shape
    X_tr = sc.fit_transform(X_tr.reshape(-1, ch)).reshape(X_tr.shape)
    X_va = sc.transform(X_va.reshape(-1, ch)).reshape(X_va.shape)

    model = build_model((WIN_SIZE, N_CANAIS))

    hist = model.fit(
        X_tr, to_categorical(y_tr, N_CLASSES),
        validation_data=(X_va, to_categorical(y_va, N_CLASSES)),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
        ],
        verbose=1
    )
    histories.append(hist)

    y_prob = model.predict(X_va, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_va, y_pred)
    f1  = f1_score(y_va, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_va, y_pred, average='weighted', zero_division=0)
    auc = calcular_auc(y_va, y_prob)

    results.append({'fold': fold+1, 'accuracy': acc, 'f1': f1, 'recall': rec, 'auc': auc})
    all_y_true.extend(y_va.tolist())
    all_y_pred.extend(y_pred.tolist())

    print(f'Acc={acc:.4f} | F1={f1:.4f} | Recall={rec:.4f} | AUC={auc:.4f}')

    del model, X_tr, X_va
    gc.collect()
    tf.keras.backend.clear_session()

# resultados
df = pd.DataFrame(results)
print('\nResultados por fold:')
print(df.to_string(index=False))
print()
for col in ['accuracy', 'f1', 'recall', 'auc']:
    print(f'{col}: {df[col].mean():.4f} +/- {df[col].std():.4f}')

df.to_csv(os.path.join(BASE_DIR, 'resultados_CNN_LSTM.csv'), index=False)

# curvas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cores = plt.cm.tab10(np.linspace(0, 1, N_FOLDS))
for i, h in enumerate(histories):
    axes[0].plot(h.history['accuracy'],     color=cores[i], label=f'Treino F{i+1}')
    axes[0].plot(h.history['val_accuracy'], color=cores[i], ls='--', alpha=0.7, label=f'Val F{i+1}')
    axes[1].plot(h.history['loss'],         color=cores[i], label=f'Treino F{i+1}')
    axes[1].plot(h.history['val_loss'],     color=cores[i], ls='--', alpha=0.7, label=f'Val F{i+1}')
axes[0].set_title('Acuracia'); axes[0].set_xlabel('Epocas'); axes[0].legend(fontsize=7)
axes[1].set_title('Loss');     axes[1].set_xlabel('Epocas'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'curvas_cnn_lstm.png'), dpi=150)
plt.show()

# matriz de confusao
cm      = confusion_matrix(all_y_true, all_y_pred, labels=np.arange(N_CLASSES))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
lbls    = [f'G{i+1:02d}' for i in range(N_CLASSES)]
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues', ax=axes[0], xticklabels=lbls, yticklabels=lbls)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1], xticklabels=lbls, yticklabels=lbls)
axes[0].set_title('Matriz de Confusao - Contagem')
axes[1].set_title('Matriz de Confusao - Normalizada')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'confusao_cnn_lstm.png'), dpi=150)
plt.show()

print('\nFim!')

2026-05-03 09:47:23.711346: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 09:47:23.736177: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-03 09:47:23.850524: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-03 09:47:23.850602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-03 09:47:23.870194: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Carregando labels...
  S1: 96320 janelas
  S2: 145740 janelas
  S3: 96320 janelas
Total: 338380 janelas

---- Fold 1/3 ----
Carregando treino (225586 amostras)...
Carregando validacao (112794 amostras)...


2026-05-03 09:53:10.026387: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-03 09:53:10.032733: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/10
7050/7050 [==============================] - 695s 98ms/step - loss: 1.6340 - accuracy: 0.4271 - val_loss: 1.0678 - val_accuracy: 0.6121 - lr: 0.0010
Epoch 2/10
7050/7050 [==============================] - 694s 98ms/step - loss: 1.0492 - accuracy: 0.6319 - val_loss: 0.8267 - val_accuracy: 0.6995 - lr: 0.0010
Epoch 3/10
 205/7050 [..............................] - ETA: 7:39 - loss: 0.9377 - accuracy: 0.6748